In [19]:
import pandas as pd

labeled = pd.read_parquet("../data/processed/labeled_stations.parquet")
print(labeled.shape)
labeled.head()

(145171, 10)


,Unnamed: 0,time,name,WSE,code,station_clean,Danger Level (m),Warning Level (m),HFL / Highest Flood Level (m),risk_label
0,0,2020-01-01 08:00:00,Baltara,30.59,024-MGD4PTN,Baltara,33.85,32.85,36.4,normal
1,1,2020-01-01 13:00:00,Baltara,30.59,024-MGD4PTN,Baltara,33.85,32.85,36.4,normal
2,2,2020-01-01 18:00:00,Baltara,30.58,024-MGD4PTN,Baltara,33.85,32.85,36.4,normal
3,3,2020-01-02 08:00:00,Baltara,30.58,024-MGD4PTN,Baltara,33.85,32.85,36.4,normal
4,4,2020-01-02 13:00:00,Baltara,30.58,024-MGD4PTN,Baltara,33.85,32.85,36.4,normal


In [20]:
def resample_station(df, freq="6h"):
    df = df.set_index("time").sort_index()
    numeric_cols = ["WSE"]
    static_cols = ["Danger Level (m)", "Warning Level (m)", "HFL / Highest Flood Level (m)", "station_clean"]
    
    out = df[numeric_cols].resample(freq).mean().interpolate(limit=2)
    for c in static_cols:
        out[c] = df[c].iloc[0]
    return out.reset_index()

resampled_list = []
for station in labeled["station_clean"].unique():
    station_df = labeled[labeled["station_clean"] == station]
    resampled_list.append(resample_station(station_df))

resampled = pd.concat(resampled_list, ignore_index=True)

print(resampled.shape)
print(resampled["time"].diff().value_counts().head())

(49145, 6)
time
0 days 06:00:00         49140
-2457 days +00:00:00        4
Name: count, dtype: int64


In [21]:
def label_risk(row):
    wse = row["WSE"]
    warning = row["Warning Level (m)"]
    danger = row["Danger Level (m)"]
    hfl = row["HFL / Highest Flood Level (m)"]
    if pd.isna(wse) or pd.isna(danger):
        return "unknown"
    elif wse >= hfl:
        return "extreme"
    elif wse >= danger:
        return "danger"
    elif wse >= warning:
        return "warning"
    else:
        return "normal"

resampled["risk_label"] = resampled.apply(label_risk, axis=1)
resampled = resampled.dropna(subset=["WSE"])  # drop rows the interpolation couldn't fill
resampled["risk_label"] = resampled["risk_label"].replace("extreme", "danger")

print(resampled.shape)
print(resampled["risk_label"].value_counts())

(48677, 7)
risk_label
normal     32908
warning     8888
danger      6881
Name: count, dtype: int64


In [22]:

resampled = resampled.sort_values(["station_clean", "time"]).reset_index(drop=True)
grp = resampled.groupby("station_clean")["WSE"]

resampled["wse_change_1step"] = grp.diff()
resampled["wse_change_4step"] = grp.diff(4)
resampled["wse_change_20step"] = grp.diff(20)
resampled["wse_rolling_std_4"] = grp.transform(lambda x: x.rolling(4, min_periods=1).std()).fillna(0)
resampled["wse_rolling_std_20"] = grp.transform(lambda x: x.rolling(20, min_periods=1).std()).fillna(0)

# New: distance to thresholds — legitimate here since we predict a FUTURE label, not the current one
resampled["dist_to_warning_boundary"] = resampled["Warning Level (m)"] - resampled["WSE"]
resampled["dist_to_danger_boundary"] = resampled["Danger Level (m)"] - resampled["WSE"]

feature_cols = [
    "wse_change_1step", "wse_change_4step", "wse_change_20step",
    "wse_rolling_std_4", "wse_rolling_std_20",
    "dist_to_warning_boundary", "dist_to_danger_boundary"
]
resampled[feature_cols] = resampled[feature_cols].fillna(0)

resampled[["station_clean", "time", "WSE"] + feature_cols].head(10)



,station_clean,time,WSE,wse_change_1step,wse_change_4step,wse_change_20step,wse_rolling_std_4,wse_rolling_std_20,dist_to_warning_boundary,dist_to_danger_boundary
0,Baltara,2020-01-01 06:00:00,30.590,0.000,0.000,0.0,0.000000e+00,0.000000,2.260,3.260
1,Baltara,2020-01-01 12:00:00,30.590,0.000,0.000,0.0,0.000000e+00,0.000000,2.260,3.260
2,Baltara,2020-01-01 18:00:00,30.580,-0.010,0.000,0.0,5.773503e-03,0.005774,2.270,3.270
3,Baltara,2020-01-02 00:00:00,30.580,0.000,0.000,0.0,5.773503e-03,0.005774,2.270,3.270
4,Baltara,2020-01-02 06:00:00,30.580,0.000,-0.010,0.0,5.000000e-03,0.005477,2.270,3.270
5,Baltara,2020-01-02 12:00:00,30.580,0.000,-0.010,0.0,7.367352e-09,0.005164,2.270,3.270
6,Baltara,2020-01-02 18:00:00,30.580,0.000,0.000,0.0,7.367352e-09,0.004880,2.270,3.270
7,Baltara,2020-01-03 00:00:00,30.575,-0.005,-0.005,0.0,2.500000e-03,0.005303,2.275,3.275
8,Baltara,2020-01-03 06:00:00,30.570,-0.005,-0.010,0.0,4.787136e-03,0.006346,2.280,3.280
9,Baltara,2020-01-03 12:00:00,30.570,0.000,-0.010,0.0,4.787136e-03,0.006852,2.280,3.280


In [23]:
horizons = {"24h": 4, "3day": 12, "5day": 20}

for name, steps in horizons.items():
    resampled[f"future_risk_{name}"] = resampled.groupby("station_clean")["risk_label"].shift(-steps)

print(resampled[[f"future_risk_{n}" for n in horizons]].notna().sum())

future_risk_24h     48657
future_risk_3day    48617
future_risk_5day    48577
dtype: int64


In [24]:
horizons = {"24h": 4, "3day": 12, "5day": 20}

for name, steps in horizons.items():
    resampled[f"future_risk_{name}"] = resampled.groupby("station_clean")["risk_label"].shift(-steps)

print(resampled[[f"future_risk_{n}" for n in horizons]].notna().sum())

future_risk_24h     48657
future_risk_3day    48617
future_risk_5day    48577
dtype: int64


In [28]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import pandas as pd

trend_features = ["wse_change_1step", "wse_change_4step", "wse_change_20step", "wse_rolling_std_4", "wse_rolling_std_20"]
trend_plus_threshold_features = trend_features + ["dist_to_warning_boundary", "dist_to_danger_boundary"]

horizon_feature_sets = {
    "24h": trend_features,                     # trend-only: stays a genuine forecasting task
    "3day": trend_plus_threshold_features,      # trend + threshold proximity: drift has time to matter
    "5day": trend_plus_threshold_features,
}

results = {}

for horizon_name in horizons:
    target_col = f"future_risk_{horizon_name}"
    cols_for_this_horizon = horizon_feature_sets[horizon_name]
    
    data = resampled.dropna(subset=[target_col]).copy()
    
    X = pd.get_dummies(data[cols_for_this_horizon + ["station_clean"]], columns=["station_clean"])
    y = data[target_col]
    time_col = data["time"]
    
    X_train = X[time_col < "2025-01-01"]
    X_test = X[time_col >= "2025-01-01"]
    y_train = y[time_col < "2025-01-01"]
    y_test = y[time_col >= "2025-01-01"]
    
    model = RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1, class_weight="balanced")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    print(f"\n===== Horizon: {horizon_name} (features: {cols_for_this_horizon}) =====")
    print(classification_report(y_test, y_pred))
    
    results[horizon_name] = {"model": model, "X_train_cols": X_train.columns, "feature_cols": cols_for_this_horizon}


===== Horizon: 24h (features: ['wse_change_1step', 'wse_change_4step', 'wse_change_20step', 'wse_rolling_std_4', 'wse_rolling_std_20']) =====
              precision    recall  f1-score   support

      danger       0.53      0.79      0.63      2001
      normal       0.96      0.73      0.83      8571
     warning       0.45      0.69      0.55      2018

    accuracy                           0.73     12590
   macro avg       0.65      0.74      0.67     12590
weighted avg       0.81      0.73      0.75     12590


===== Horizon: 3day (features: ['wse_change_1step', 'wse_change_4step', 'wse_change_20step', 'wse_rolling_std_4', 'wse_rolling_std_20', 'dist_to_warning_boundary', 'dist_to_danger_boundary']) =====
              precision    recall  f1-score   support

      danger       0.92      0.91      0.91      2001
      normal       0.98      0.94      0.96      8539
     warning       0.72      0.83      0.77      2010

    accuracy                           0.92     12550
   ma

In [26]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

for horizon_name, r in results.items():
    joblib.dump(r["model"], f"../models/model_{horizon_name}.pkl")
    joblib.dump(list(r["X_train_cols"]), f"../models/columns_{horizon_name}.pkl")
    joblib.dump(r["feature_cols"], f"../models/features_{horizon_name}.pkl")  # new: track which features this horizon uses

print("Saved models:", list(results.keys()))

Saved models: ['24h', '3day', '5day']


In [27]:
os.makedirs("../data/processed", exist_ok=True)
resampled.to_parquet("../data/processed/resampled_stations.parquet", index=False)
print("Saved:", resampled.shape)

Saved: (48677, 17)
